# GBDT as a Kernel: Finding Where Your Model Is Weak

*Companion notebook for the MRM **Tabular** series.*

A gradient-boosted tree ensemble quietly defines a **kernel**: two rows are similar if they
fall in the **same leaves** across the trees. That leaf co-membership matrix *is* a kernel `K`.

In this notebook we, using **Modeva** end to end:

1. Load **California Housing** and split it into train / test.
2. Build an XGBoost model by **cross-validation** on the training data.
3. Treat that GBDT as a **kernel** (leaf co-membership).
4. Use a **Nystrom spectral embedding** of the kernel to cluster the data by proximity.
5. Read off the model's **performance per cluster** to find where it is weak, and plot it as a
   **MoCharts** bar chart.

## Setup

Modeva is licence-gated. Generate a free licence at [modeva.ai](https://modeva.ai), then
uncomment and run the cell below (needed on Colab / a fresh environment).

In [ ]:
# !pip install -q modeva
# from modeva.utils.authenticate import authenticate
# authenticate(auth_code="PASTE_YOUR_LICENCE_CODE_HERE")

In [ ]:
import numpy as np
from modeva import DataSet, TestSuite
from modeva.models import MoXGBRegressor, MoFuseKernelRegressor, ModelTuneGridSearch

## 1. Load the data

Modeva ships California Housing as a built-in dataset. `load` also sets the regression target
(`MedHouseVal`) and the feature schema for us.

In [ ]:
ds = DataSet()
ds.load(name="CaliforniaHousing")
print("target :", ds.target_feature_name)
print("features:", list(ds.feature_names))

## 2. Train / test split

Split the **full** dataset 80/20 with Modeva's `set_random_split`. There is no need to subsample:
the kernel ridge fit runs on a bounded **Nystrom** support set, so it scales roughly *linearly in
`n`* rather than the naive `O(n^3)` of a dense kernel solve.

In [ ]:
ds.set_random_split(test_ratio=0.2, random_state=0)
print("train:", ds.train_x.shape, "| test:", ds.test_x.shape)

## 3. Preprocess

Min-max scale the numerical features through Modeva's preprocessing pipeline so the split shares
one consistent transform.

In [ ]:
ds.scale_numerical(method="minmax")
ds.preprocess()

## 4. Build the model by cross-validation

Tune an XGBoost regressor with Modeva's `ModelTuneGridSearch` using **5-fold cross-validation on
the training data**. We rank by MSE and keep the best configuration.

In [ ]:
base = MoXGBRegressor(max_depth=3, verbosity=0)
hpo = ModelTuneGridSearch(dataset=ds, model=base)
cv_result = hpo.run(
    param_grid={"n_estimators": [100, 200, 300], "learning_rate": [0.03, 0.05, 0.1]},
    metric=("MSE", "MAE", "R2"),
    cv=5,
)
cv_result.table

In [ ]:
best_params = cv_result.value["params"][0]
best_params

Retrain on the full training set with the best hyperparameters and check overall test performance
through the `TestSuite`.

In [ ]:
xgb = MoXGBRegressor(**best_params, name="XGB-Tuned", verbosity=0)
xgb.fit(ds.train_x, ds.train_y.ravel())
ts = TestSuite(ds, xgb)
ts.diagnose_accuracy_table().table

## 5. GBDT as a kernel

Two points that land in the **same leaves** across the boosting rounds are treated as similar; the
leaf co-membership matrix is a valid kernel. In Modeva we obtain exactly this kernel with a
`MoFuseKernelRegressor` restricted to the **XGBoost channel only** — `use_rbf=False`,
`use_spectral=False` — so there is no RBF or learned spectral component, just the pure GBDT
leaf-kernel. We pass the **cross-validated hyperparameters** so the kernel comes from the same
model we tuned above. `solver="nystrom"` keeps the kernel-ridge fit on a bounded support set, so
it stays linear in the sample size on the full dataset.

In [ ]:
fk = MoFuseKernelRegressor(
    name="GBDT-Kernel",
    backend="xgboost",
    use_xgb=True, use_rbf=False, use_spectral=False,  # pure leaf co-membership kernel
    solver="nystrom",
    gbdt_params=best_params,
    random_state=0,
)
fk.fit(ds.train_x, ds.train_y.ravel())
print("pure GBDT leaf-kernel (general path off):", not getattr(fk.estimator_, "_general", False))

## 6. Nystrom spectral clustering → model weakness

`diagnose_weak_clusters` builds a degree-normalised **Nystrom** spectral embedding of that kernel
(landmarks keep the eigensolve linear in `n`), clusters the data in the embedding, and reports the
model metric **per cluster** on train and test. Regions with a low metric or a large train/test
gap are where the model is weak.

In [ ]:
weak = fk.diagnose_weak_clusters(ds, n_clusters=5)
weak.table

## 7. Per-cluster performance (MoCharts bar chart)

`plot()` renders a MoCharts bar chart of the per-cluster test metric — the shortest bars are the
clusters the model handles worst.

In [ ]:
weak.plot()

And the explicit weakest-cluster ranking:

In [ ]:
weak.value["worst_clusters"]

## 8. What characterizes the weak region?

A weak cluster is only actionable if we know *where* it sits in feature space.
`diagnose_weak_clusters` already computes this for you: the result carries the per-cluster
feature profile and the weakest cluster's *signature* -- no manual post-processing needed.

`cluster_feature_means` is the mean of each (min-max scaled) feature per cluster:

In [ ]:
weak.value["cluster_feature_means"]

And `weak_cluster_signature` ranks how the weakest cluster deviates from the overall mean --
the features that set the weak region apart, and the starting point for a targeted fix (more
data, a local expert model, or feature work in that region):

In [ ]:
print("Weakest cluster:", weak.value["weak_cluster_id"])
weak.value["weak_cluster_signature"]

## Takeaways

- A GBDT is also a **kernel machine**: its leaves induce a similarity kernel over the data.
- A **Nystrom** spectral embedding turns that kernel into cheap, proximity-based clusters.
- Scoring the model **per cluster** produces a *weakness map* — a principled, model-native way to
  see *where* an accurate-on-average model actually fails, which is exactly what you want before
  targeted repair (mixture-of-experts, residual models, or more data in the weak regions).